In [4]:
#!/usr/bin/env python3
import json
import shutil
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional

ENCODING = "utf-8"

def crc_sidecar(path: Path) -> Path:
    """
    Hadoop local filesystem checksum sidecar:
      file:  metadata.json
      crc:   .metadata.json.crc
    """
    return path.with_name(f".{path.name}.crc")


def delete_crc_for(path: Path, verbose: bool = True) -> None:
    crc = crc_sidecar(path)
    if crc.exists():
        crc.unlink()
        if verbose:
            print(f"DELETED-CRC: {crc}")


def delete_crc_tree(root: Path, verbose: bool = True) -> int:
    count = 0
    for crc in root.rglob(".*.crc"):
        crc.unlink()
        count += 1
        if verbose:
            print(f"DELETED-CRC: {crc}")
    return count

def backup_once(path: Path) -> Path:
    bak = path.with_suffix(path.suffix + ".bak")
    if not bak.exists():
        shutil.copy2(path, bak)
    return bak

def walk_replace(obj: Any, old_prefix: str, new_prefix: str) -> Tuple[Any, int]:
    """Recursively replace old_prefix in any string; return (new_obj, num_changes)."""
    changes = 0
    if isinstance(obj, dict):
        out = {}
        for k, v in obj.items():
            nv, c = walk_replace(v, old_prefix, new_prefix)
            out[k] = nv
            changes += c
        return out, changes
    elif isinstance(obj, list):
        out = []
        for v in obj:
            nv, c = walk_replace(v, old_prefix, new_prefix)
            out.append(nv)
            changes += c
        return out, changes
    elif isinstance(obj, str):
        if old_prefix in obj:
            return obj.replace(old_prefix, new_prefix), 1
        return obj, 0
    else:
        return obj, 0

def _rewrite_json_struct(data: Any, old_prefix: str, new_prefix: str) -> Tuple[Any, int, int]:
    """Return (new_data, records_changed, fields_changed). For JSON, 'records' means top-level items touched."""
    new_data, fields_changed = walk_replace(data, old_prefix, new_prefix)
    # define 'records_changed' heuristically:
    # - list top-level: count elements that had any change
    # - dict or scalar: 1 if any field changed else 0
    if isinstance(data, list):
        # recompute per-element changes to count 'records'
        records_changed = 0
        for old, new in zip(data, new_data):
            _, c = walk_replace(old, old_prefix, new_prefix)
            records_changed += int(c > 0)
    else:
        records_changed = int(fields_changed > 0)
    return new_data, records_changed, fields_changed

def rewrite_json_file(path: Path, old_prefix: str, new_prefix: str, apply: bool, verbose: bool) -> Tuple[int, int]:
    """
    Rewrite JSON or JSONL/NDJSON file.
    Returns (records_changed, fields_changed).
    """
    suffix = path.suffix.lower()
    records_changed_total = 0
    fields_changed_total = 0

    if suffix in (".json",):
        with path.open("r", encoding=ENCODING) as f:
            data = json.load(f)

        new_data, rec_chg, fld_chg = _rewrite_json_struct(data, old_prefix, new_prefix)
        records_changed_total += rec_chg
        fields_changed_total += fld_chg

        if fld_chg and apply:
            backup_once(path)
            tmp = path.with_suffix(path.suffix + ".tmp")
            try:
                with tmp.open("w", encoding=ENCODING) as out:
                    json.dump(new_data, out, ensure_ascii=False)
                tmp.replace(path)
                delete_crc_for(path, verbose=verbose)
                if verbose:
                    print(f"UPDATED: {path} (+{fld_chg} field changes across {rec_chg} records)")
            finally:
                if tmp.exists():
                    try:
                        tmp.unlink()
                    except Exception:
                        pass
        elif fld_chg and not apply and verbose:
            print(f"WOULD-UPDATE: {path} (+{fld_chg} field changes across {rec_chg} records)")

    elif suffix in (".jsonl", ".ndjson"):
        # line-delimited JSON
        lines_out: List[str] = []
        changed_any = False

        with path.open("r", encoding=ENCODING) as f:
            for line in f:
                stripped = line.strip()
                if not stripped:
                    lines_out.append(line)
                    continue
                try:
                    obj = json.loads(line)
                except json.JSONDecodeError:
                    # non-JSON line: pass through
                    lines_out.append(line)
                    continue

                new_obj, rec_chg, fld_chg = _rewrite_json_struct(obj, old_prefix, new_prefix)
                records_changed_total += rec_chg
                fields_changed_total += fld_chg
                if fld_chg:
                    changed_any = True
                lines_out.append(json.dumps(new_obj, ensure_ascii=False) + "\n")

        if changed_any and apply:
            backup_once(path)
            tmp = path.with_suffix(path.suffix + ".tmp")
            try:
                with tmp.open("w", encoding=ENCODING) as out:
                    out.writelines(lines_out)
                tmp.replace(path)
                delete_crc_for(path, verbose=verbose)
                if verbose:
                    print(f"UPDATED: {path} (+{fields_changed_total} field changes across {records_changed_total} records)")
            finally:
                if tmp.exists():
                    try:
                        tmp.unlink()
                    except Exception:
                        pass
        elif changed_any and not apply and verbose:
            print(f"WOULD-UPDATE: {path} (+{fields_changed_total} field changes across {records_changed_total} records)")
    else:
        # ignore other file types
        return 0, 0

    return records_changed_total, fields_changed_total

def run_update_json(root: Path, old_prefix: str, new_prefix: str, apply: bool, verbose: bool = True) -> None:
    total_files = 0
    changed_files = 0
    total_rec = 0
    total_fields = 0

    patterns = ["metadata/*.json", "metadata/*.jsonl", "metadata/*.ndjson"]
    for pat in patterns:
        for path in root.rglob(pat):
            total_files += 1
            try:
                rec_chg, fld_chg = rewrite_json_file(path, old_prefix, new_prefix, apply, verbose)
                if fld_chg:
                    changed_files += 1
                    total_rec += rec_chg
                    total_fields += fld_chg
            except Exception as e:
                print(f"# WARN: failed to process {path}: {e}")

    print("\nSummary:")
    print(f"  JSON files scanned : {total_files}")
    print(f"  Files with changes : {changed_files}")
    print(f"  Records changed    : {total_rec}")
    print(f"  Field updates      : {total_fields}")
    print("  Mode               :", "APPLY (writes performed, .bak created)" if apply else "DRY-RUN (no writes)")

In [6]:
ROOT = Path(r"/mnt/iceberg-ldbc-snb-1tb/warehouse/ldbc_cnb_sf1000.db") 
OLD_PREFIX = r"/mnt/iceberg/warehouse/ldbc_cnb_sf1000.db/"
NEW_PREFIX = "/mnt/iceberg/warehouse/ldbc_snb_sf1000.db/"      

APPLY = False
VERBOSE = True
ENCODING = "utf-8"

run_update_json(ROOT, OLD_PREFIX, NEW_PREFIX, APPLY, verbose=VERBOSE)


UPDATED: /mnt/iceberg-ldbc-snb-1tb/warehouse/ldbc_cnb_sf1000.db/forum_has_member_person/metadata/00000-0d557e29-0c67-41d6-8c1b-b10bc7eb45f2.metadata.json (+2 field changes across 1 records)
UPDATED: /mnt/iceberg-ldbc-snb-1tb/warehouse/ldbc_cnb_sf1000.db/person_likes_comment/metadata/00000-36674d48-a014-4a72-930d-7fadccc9faf4.metadata.json (+2 field changes across 1 records)
UPDATED: /mnt/iceberg-ldbc-snb-1tb/warehouse/ldbc_cnb_sf1000.db/forum_has_tag_tag/metadata/00000-79a37793-24d5-47dc-8bd6-c59e836f1dd6.metadata.json (+2 field changes across 1 records)
UPDATED: /mnt/iceberg-ldbc-snb-1tb/warehouse/ldbc_cnb_sf1000.db/organisation/metadata/00000-a1b35c45-249c-41d7-8059-85a9f0365c18.metadata.json (+2 field changes across 1 records)
UPDATED: /mnt/iceberg-ldbc-snb-1tb/warehouse/ldbc_cnb_sf1000.db/post/metadata/00000-e2d39203-0028-4de5-acfc-f31d69f5443c.metadata.json (+2 field changes across 1 records)
UPDATED: /mnt/iceberg-ldbc-snb-1tb/warehouse/ldbc_cnb_sf1000.db/person_knows_person/metad